## SMART_ATTENDANCE PROJECT

In [6]:
import pandas as pd
df=pd.read_excel(r"C:\Users\vv\OneDrive\Desktop\PYTHON\SMART_ATTENDANCE\ATTENDANCE(SMART) OPEN_CV.xlsx")
import cv2

In [7]:
face_cascade=cv2.CascadeClassifier(r"C:\Users\vv\OneDrive\Desktop\PYTHON\haarcascade_frontalface_default.xml")
cam=cv2.VideoCapture(0)

#### TAKE A PHOTO OF USER ( VIA Id AND NAME)

In [30]:
import cv2
import pandas as pd
import os

# ---------------- Excel Part ----------------

Id = input("Enter your Id : ")
Name = input("Enter your Name : ")

excel_path = r"C:\Users\vv\OneDrive\Desktop\PYTHON\SMART_ATTENDANCE\ATTENDANCE(SMART) OPEN_CV.xlsx"

# Read existing excel
if os.path.exists(excel_path):
    df = pd.read_excel(excel_path)
else:
    df = pd.DataFrame(columns=["Id", "Name"])

# Add new data
df2 = pd.DataFrame({"Id": [Id], "Name": [Name]})

# Combine
df = pd.concat([df, df2], ignore_index=True)

# Remove duplicates
df = df.drop_duplicates().reset_index(drop=True)

# Save excel
try:
    df.to_excel(excel_path, index=False)
    print("Data saved successfully.")
except PermissionError:
    print("Close the Excel file first.")
    exit()

# ---------------- Camera Part ----------------

cam = cv2.VideoCapture(0)

if not cam.isOpened():
    print("Camera not opening")
    exit()

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

sampleNum = 0

while True:

    ret, img = cam.read()

    if not ret:
        print("Failed to capture image")
        break

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:

        cv2.rectangle(img, (x, y), (x+w, y+h), (255, 0, 0), 2)

        sampleNum += 1

        # Save face image
        cv2.imwrite(r"C:\Users\vv\OneDrive\Desktop\PYTHON\SMART_ATTENDANCE\faces\user."+ str(Id) + "." + str(sampleNum) + ".jpg",gray[y:y+h, x:x+w])

    # IMPORTANT: show frame outside loop
    cv2.imshow("Frame", img)

    # Exit conditions
    if cv2.waitKey(1) & 0xFF == 27:
        break

    elif sampleNum > 10:
        break

cam.release()
cv2.destroyAllWindows()

Enter your Id :  0651922352
Enter your Name :  Aayush


Data saved successfully.


#### CHECK THE STORED PHOTOS

In [31]:
import os
import cv2
import numpy as np
from PIL import Image

# Initialize the recognizer
recognizer = cv2.face.LBPHFaceRecognizer_create()

# FIX: Corrected folder path (ensure this folder actually exists)
path = r"C:\Users\vv\OneDrive\Desktop\PYTHON\SMART_ATTENDANCE\faces"

def getImagesWithID(path):
    # Get all file paths in the directory
    imagePaths = [os.path.join(path, f) for f in os.listdir(path)]
    faces = []
    IDs = []
    
    for imagePath in imagePaths:
        # Skip system files like Thumbs.db if they exist
        if not imagePath.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
            
        # FIX: Added quotes around 'L' for grayscale conversion
        faceImg = Image.open(imagePath).convert('L')
        
        # FIX: Added quotes around 'uint8' data type
        faceNp = np.array(faceImg, 'uint8')
        
        # Extracts ID assuming filename format like "user.1.jpg"
        ID = int(os.path.split(imagePath)[-1].split('.')[1])
        
        faces.append(faceNp)
        IDs.append(ID)
        
        cv2.imshow("training", faceNp)
        cv2.waitKey(100) 
        
    return np.array(IDs), faces

# Run the training process
Ids, faces = getImagesWithID(path)
recognizer.train(faces, Ids)
recognizer.save(r"C:\Users\vv\OneDrive\Desktop\PYTHON\SMART_ATTENDANCE\trainningData.yml")
cv2.destroyAllWindows()

#### BY DEFAULT FEED AS ABSENT AND FILL THE DATE

In [42]:
import pandas as pd
df=pd.read_excel(r"C:\Users\vv\OneDrive\Desktop\PYTHON\SMART_ATTENDANCE\ATTENDANCE(SMART) OPEN_CV.xlsx")
date=input("Enter date- dd/mm/yyyy")
df[date]='A'
df.to_excel(r"C:\Users\vv\OneDrive\Desktop\PYTHON\SMART_ATTENDANCE\ATTENDANCE(SMART) OPEN_CV.xlsx", index=False)
df

Enter date- dd/mm/yyyy 02/06/2026


,Id,Name,02/06/2026
0,1,Anuj,A
1,2,nick,A
2,3,sam,A
3,55,Advik,A
4,28,Abhi Tyagi,A
5,651922352,Aayush,A


#### NAME VERIFICATION VIA PHOTO AND AUTO WILL SET USER PRESENT(AFTER VERIFIED BY PHOTO)

In [41]:
import cv2 
import numpy as np 
import pandas as pd 

date = input("Enter date-dd/mm/yyyy: ") 
faceDetect = cv2.CascadeClassifier(r'C:\Users\vv\OneDrive\Desktop\PYTHON\haarcascade_frontalface_default.xml') 
cam = cv2.VideoCapture(0) 

rec = cv2.face.LBPHFaceRecognizer_create() 
rec.read(r'C:\Users\vv\OneDrive\Desktop\PYTHON\SMART_ATTENDANCE\trainningData.yml') 

id = 0 
font = cv2.FONT_HERSHEY_COMPLEX_SMALL 
excel_path = r'C:\Users\vv\OneDrive\Desktop\PYTHON\SMART_ATTENDANCE\ATTENDANCE(SMART) OPEN_CV.xlsx'
df = pd.read_excel(excel_path) 

while True: 
    ret, img = cam.read() 
    if not ret: # Prevents crashing if the camera feed fails
        break
        
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) # Fixed COLOR_BGRA2GRAY to BGR
    faces = faceDetect.detectMultiScale(gray, 2, 4) 
    
    for (x, y, w, h) in faces: 
        cv2.rectangle(img, (x, y), (x+w, y+h), (0, 0, 255), 2) 
        id, conf = rec.predict(gray[y:y+h, x:x+w]) 
        
        # FIXED: Extracting scalar value from pandas Series
        name_series = df['Name'][df.Id == id]
        id1 = name_series.iloc[0] if not name_series.empty else "Unknown"
        
        # FIXED: Corrected syntax to assign the 'P' value to the specific cell
        if date in df.columns:
            df.loc[df.Id == id, date] = 'P' 
        
        df.to_excel(excel_path, index=False) 
        
        cv2.putText(img, str(id1), (x, y+h), font, 1, (0, 255, 0), 2) # Fixed font scaling
        
    cv2.imshow('Face', img) 
    
    # 27 is the ASCII code for the 'ESC' key
    if (cv2.waitKey(1) & 0xFF == 27): 
        break 

cam.release() 
cv2.destroyAllWindows()

Enter date-dd/mm/yyyy:  02/06/2026
